# 02 — Pricing Desk (live rate sheet)
Pricing is ACTIVE: numbers and rules change here, not in engine code. Edit the RateCard cell, re-run the quote cells, watch LR move.

**Connector (API shape):** request = simulated book + regime name + RateCard → `quote(book, regime, card)` → response = book + `FINAL_PREMIUM_SST`. Simulation notebooks only ever call `quote` / `price_many`; they never implement pricing.

**N regimes:** any name in `PRICERS` can be quoted. Add one via the standard format (`func(book, card)`, `register_pricer`), then include it in any `price_many` call. Tables and figures take however many books you hand them.

Per-method sheets (own rules, own card, same API output): `02a_tariff.ipynb` · `02b_glm.ipynb` · `02c_telem.ipynb`. This notebook stays the comparison hub.

In [ ]:
import copy, os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

import pandas as pd
from voltvision import CFG, SCEN, N_YEARS, REGIMES, PRICERS
from voltvision import RateCard, quote, price_many, register_pricer
from voltvision import ensure_core_methods
from voltvision import summary, lr, lr_by, gen, simulate
from voltvision import io
ensure_core_methods()  # load CALC cells from 02a/02b/02c
print('registered pricers:', sorted(PRICERS))


## Rate card (EDIT ME — every live pricing number)
Change values, re-run the quote cells below. `from_cfg` starts from `CFG`; overrides win.

In [ ]:
card = RateCard.from_cfg(CFG)
# --- tweak here, e.g.: ---
# card.expense_loading = 1.8
# card.tpo_loading = 1.50
# card.sst = 0.10
display(pd.DataFrame(vars(card).items(), columns=['field', 'value']))


## Load one simulated book (fallback: quick inline sim if `01` not run yet)

In [ ]:
SC = "MIX"  # scenario to price
try:
    book = io.load_sim(SC)
    print(f"loaded shared sim_{SC}: {len(book)} rows")
except FileNotFoundError:
    print('shared book missing — quick inline sim')
    c = copy.deepcopy(CFG)
    c['n'] = 2000
    book = simulate(gen(c, SCEN[SC], c['seed']), c, SCEN[SC],
                    seed=c['seed'], n_years=N_YEARS, verbose=False)


## Quote (same book, every regime, side by side)

In [ ]:
books = {m: quote(book, m, card) for m in REGIMES}
display(summary(books))
print('LR by coverage (rows=coverage, cols=regime):')
display(pd.DataFrame({m: lr_by(b, 'COVERAGE_TYPE').round(1) for m, b in books.items()}))
print('LR by vehicle:')
display(pd.DataFrame({m: lr_by(b, 'VEHICLE_TYPE').round(1) for m, b in books.items()}))


## What-if (tweak card, re-quote, compare LR)

In [ ]:
alt = RateCard.from_cfg(CFG, expense_loading=1.8, tpo_loading=1.50)
base_lr = {m: round(lr(b), 1) for m, b in books.items()}
alt_books = {m: quote(book, m, alt) for m in REGIMES}
alt_lr = {m: round(lr(b), 1) for m, b in alt_books.items()}
display(pd.DataFrame({'base LR (%)': base_lr, 'alt LR (%)': alt_lr,
    'delta (pp)': {m: round(alt_lr[m] - base_lr[m], 1) for m in base_lr}}))
print('alt card: expense_loading=1.8, tpo_loading=1.50')


## Nth regime (standard format demo — copy this pattern for real ones)
Any `func(book, card)` returning `FINAL_PREMIUM_SST` can join the desk.

In [ ]:
def price_floor_demo(book, card):
    """DEMO: tariff with a minimum premium floor of RM800."""
    out = quote(book, 'tariff', card)
    out['FINAL_PREMIUM_SST'] = out['FINAL_PREMIUM_SST'].clip(lower=800.0)
    return out

register_pricer('floor_demo', price_floor_demo)
books4 = price_many(book, CFG, [*REGIMES, 'floor_demo'])
display(summary(books4))
print('desk now holds', len(books4), 'regimes; summary/figures take N')


In [ ]:
for m, b in books.items():
    p = io.save_priced(SC, m, b)
    print(f"{m}: {len(b)} rows -> {p}")


## Notes / improvement backlog
- TPO tariff prices below expected cost (TPO LR >100%) — move `tpo_sa_pct` / `tpo_loading` on the card above when approved.
- GLM/telem train out-of-sample by default (seed 42, window 2021–2025); in-sample legacy via `train_book_seed=None` (leak size in `04 §5c`).
- Next: `03_main.ipynb` runs sim + pricing for all scenarios with figures and Excel export.